## This file is the data analysis for Austin Property value over several years
Data source: https://datausa.io/profile/geo/austin-tx/?race-income-income_geo=incomeRace5

In [ ]:
# Import Necessary Libraries
import numpy as np
import pandas as pd

In [ ]:
# Load in data sets

# Set names of data set files for local access
path_1 = '../data/austin_property_value.csv'

austin_PropVal_df = pd.read_csv(path_1)

### Sanity Checks

#### Data set 1

In [ ]:
austin_PropVal_df.shape

In [ ]:
austin_PropVal_df.head(5)

In [ ]:
austin_PropVal_df.tail(5)

In [ ]:
austin_PropVal_df.info()

In [ ]:
austin_PropVal_df.describe()

### Remove unecessary columns

In [ ]:
# Make a cleaned copy
austin_PropVal_df_cleaned = austin_PropVal_df.drop(
    columns=["Value Bucket ID", "Property Value by Bucket Moe", "Place ID"]
).copy()

# Quick check
print(austin_PropVal_df_cleaned.head())
print(austin_PropVal_df_cleaned['Place'].unique())

In [ ]:
austin_PropVal_df_cleaned.head()

In [ ]:
austin_PropVal_df_cleaned.tail(5)

In [ ]:
austin_PropVal_df_cleaned.info()

In [ ]:
austin_PropVal_df_cleaned.describe()

In [ ]:
# Filter for Austin only
austin_propval = austin_PropVal_df_cleaned[austin_PropVal_df_cleaned['Place'] == "Austin, TX"].copy()

# Select quantitative columns
quant_cols = ['Year', 'Property Value by Bucket', 'share']

# Stats for Austin only
austin_stats = austin_propval[quant_cols].describe().T
austin_stats['range'] = austin_propval[quant_cols].max() - austin_propval[quant_cols].min()
austin_stats['median'] = austin_propval[quant_cols].median()

austin_stats

In [ ]:
import matplotlib.pyplot as plt

# Filter Austin only
austin_property_df = austin_PropVal_df_cleaned[austin_PropVal_df_cleaned['Place'] == "Austin, TX"].copy()

# Group by year and value bucket
grouped = austin_property_df.groupby(['Year','Value Bucket']).agg({
    'Property Value by Bucket': 'sum'
}).reset_index()

# Pivot to wide format (value buckets as columns)
pivot_df = grouped.pivot(index='Year', columns='Value Bucket', values='Property Value by Bucket').fillna(0)

# Stacked bar chart
pivot_df.plot(kind='bar', stacked=True, figsize=(12,6), colormap='tab20')
plt.ylabel("Number of Housing Units")
plt.title("Distribution of Housing Units by Property Value Bucket in Austin (2015–2023)")
plt.legend(title="Property Value Bucket", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Save austin prop_val to csv 

austin_property_df.to_csv("../data/austin_propval_cleaned.csv", index=False)

### Pivot Dataframe to index by year

In [ ]:

austin_propval_pivot = pd.pivot_table(
    austin_property_df,
    values = 'Property Value by Bucket',
    index = ['Year'],
    columns = ['Value Bucket'],
    aggfunc = 'sum'
)


### Check Dataframe adjusted correctly

In [ ]:
austin_propval_pivot.shape

In [ ]:
austin_propval_pivot.info()

In [ ]:
austin_propval_pivot.head(9)

### Combine buckets to make histogram easier to read

In [ ]:
# Create a < 99,000 bucket
austin_propval_pivot["< $99,999"] = (
    austin_propval_pivot["Less Than $10,000"] +
    austin_propval_pivot["$10,000 to $14,999"] +
    austin_propval_pivot["$15,000 to $19,999"] +
    austin_propval_pivot["$25,000 to $29,999"] +
    austin_propval_pivot["$30,000 to $34,999"] +
    austin_propval_pivot["$35,000 to $39,999"] +
    austin_propval_pivot["$40,000 to $49,999"] +
    austin_propval_pivot["$50,000 to $59,999"] +
    austin_propval_pivot["$60,000 to $69,999"] +
    austin_propval_pivot["$70,000 to $79,999"] +
    austin_propval_pivot["$80,000 to $89,999"] +
    austin_propval_pivot["$90,000 to $99,999"] 
    
)

# drop the originals 
austin_propval_pivot = austin_propval_pivot.drop(
    columns=["Less Than $10,000", "$10,000 to $14,999","$15,000 to $19,999", "$20,000 to $24,999",
             "$25,000 to $29,999", "$30,000 to $34,999", "$35,000 to $39,999", "$40,000 to $49,999",
            "$50,000 to $59,999", "$60,000 to $69,999","$70,000 to $79,999", 
             "$80,000 to $89,999","$90,000 to $99,999"]
)


# Create a  100,000 - 200,000 bucket
austin_propval_pivot["$100,000 - $199,999"] = (
    austin_propval_pivot["$100,000 to $124,999"] +
    austin_propval_pivot["$125,000 to $149,999"] +
    austin_propval_pivot["$150,000 to $174,999"] +
    austin_propval_pivot["$175,000 to $199,999"] 
)

# drop the originals 2
austin_propval_pivot = austin_propval_pivot.drop(
    columns=["$100,000 to $124,999", "$125,000 to $149,999","$150,000 to $174,999", 
             "$175,000 to $199,999"]
)
        
# Create a  200,000 - 300,000 bucketAlt
austin_propval_pivot["$200,000 - $299,999"] = (
    austin_propval_pivot["$200,000 to $249,999"] +
    austin_propval_pivot["$250,000 to $299,999"] 
)

# drop the originals 3
austin_propval_pivot = austin_propval_pivot.drop(
    columns=["$200,000 to $249,999", "$250,000 to $299,999"]
)

# Create a  1M - 2M bucket
austin_propval_pivot["$1,000,000 - $1,999,999"] = (
    austin_propval_pivot["$1,000,000 to $1,499,999"] +
    austin_propval_pivot["$1,500,000 to $1,999,999"] 
)

# drop the originals 4
austin_propval_pivot = austin_propval_pivot.drop(
    columns=["$1,000,000 to $1,499,999", "$1,500,000 to $1,999,999"]
)

### Check successfull adjustment of df

In [ ]:
austin_propval_pivot.shape

In [ ]:
austin_propval_pivot.head()

### Reorder columns by ascending order for the value buckets

In [ ]:
desired_order = [
    "< $99,999", 
    "$100,000 - $199,999",
    "$200,000 - $299,999",
    "$300,000 to $399,999",
    "$400,000 to $499,999",
    "$500,000 to $749,999",
    "$750,000 to $999,999",
    "$1,000,000 - $1,999,999",
    "$2,000,000 or More"
]

austin_propval_pivot = austin_propval_pivot[desired_order]

### Check successful reorder

In [ ]:
austin_propval_pivot.head(9)

### Graph with Altair to check data

In [ ]:
# import altair
import altair as alt

#### Yearly Stacked Bar Chart

In [ ]:

desired_order = [
    "< $99,999", "$100,000 - $199,999",
    "$200,000 - $299,999", "$300,000 to $399,999", "$400,000 to $499,999",
    "$500,000 to $749,999", "$750,000 to $999,999",
    "$1,000,000 - $1,999,999", "$2,000,000 or More"
]

# Melt to long
df_long = austin_propval_pivot.reset_index().melt(
    id_vars="Year", var_name="Bucket", value_name="Count"
)

# Stack order key
order_map = {b: i for i, b in enumerate(desired_order)}
df_long["BucketOrder"] = df_long["Bucket"].map(order_map).astype(float)

# Add year totals and ratio
df_long["YearTotal"] = df_long.groupby("Year")["Count"].transform("sum")
df_long["Ratio"] = df_long["Count"] / df_long["YearTotal"]

# Stacked bars: counts on axis, ratio in tooltip
chart = alt.Chart(df_long).mark_bar().encode(
    x=alt.X("Year:O", title="Year", axis=alt.Axis(labelAngle=-30)),
    y=alt.Y("Count:Q", stack="zero", axis=alt.Axis(title="Count of Properties", format=",")),
    color=alt.Color(
        "Bucket:N",
        title="Value Bucket",
        scale=alt.Scale(domain=desired_order)   # legend/color order
    ),
    order=alt.Order("BucketOrder:Q"),          # stack order bottom→top
    tooltip=[
        alt.Tooltip("Year:O", title="Year"),
        alt.Tooltip("Bucket:N", title="Bucket"),
        alt.Tooltip("Count:Q", title="Count", format=","),
        alt.Tooltip("YearTotal:Q", title="Year Total", format=","),
        alt.Tooltip("Ratio:Q", title="Share", format=".1%")
    ]
).properties(
    width=720, height=420, title="Austin Property Value Distribution by Year"
)

chart



#### Bar graph for each bucket, with year slider

In [ ]:

year_min = int(df_long["Year"].min())
year_max = int(df_long["Year"].max())

yr = alt.param(
    name="Year",
    bind=alt.binding_range(min=year_min, max=year_max, step=1),
    value=year_min
)

chart = (
    alt.Chart(df_long)
      .add_params(yr)
      .transform_filter(alt.datum.Year == yr)
      .mark_bar(size=30)   # ↓ thinner bars by choosing a smaller size
      .encode(
          x=alt.X("Bucket:N",
                  sort=desired_order,
                  title="Value Bucket",
                  axis=alt.Axis(labelAngle=-30)),
          y=alt.Y("Ratio:Q", title="Share of Year", 
                  axis=alt.Axis(format="%"),
                  scale = alt.Scale(domain=[0,0.3]) # fix y-axis
                 ),
          tooltip=[
              alt.Tooltip("Year:O"),
              alt.Tooltip("Bucket:N", title="Bucket"),
              alt.Tooltip("Ratio:Q", title="Share", format=".1%"),
              alt.Tooltip("Count:Q", title="Count", format=","),
              alt.Tooltip("YearTotal:Q", title="Year Total", format=","),
          ]
      )
      .properties(width=450, height=420, title="Property Value Bucket Share by Year")
)

chart


In [ ]:
#save csv files

austin_propval_pivot.to_csv("../data/austin_propval_pivot.csv", index=False)
df_long.to_csv("../data/austin_propval_pivot_long.csv", index = False)